<a href="https://colab.research.google.com/github/NavyasreeBalu/Gridlock-/blob/main/Navya_sree_Balu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install lightgbm catboost geohash2 -q --break-system-packages

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.9 MB/s eta 0:00:00


In [ ]:
!pip install geohash2 -q

In [ ]:
# 2) Imports
# ============================================
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
import geohash2

from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

RANDOM_STATE = 42
N_FOLDS = 5
np.random.seed(RANDOM_STATE)


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving sample_submission.csv to sample_submission.csv
Saving test.csv to test.csv
Saving train.csv to train.csv


In [ ]:
# ============================================
# 3) Load data
# ============================================
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")

In [ ]:
# ============================================
# 4) Feature engineering
# ============================================
def engineer(df):
    df = df.copy()

    # timestamp -> hour, minute, time slot
    ts = df["timestamp"].astype(str).str.split(":", expand=True)
    df["hour"] = ts[0].astype(int)
    df["minute"] = ts[1].astype(int)
    df["minute_block"] = (df["minute"] // 15).astype(int)
    df["time_slot"] = df["hour"] * 4 + df["minute_block"]

    # geohash -> lat/lng
    latitudes = []
    longitudes = []
    for gh in df["geohash"].astype(str):
        try:
            lat, lng = geohash2.decode(gh)
            latitudes.append(float(lat))
            longitudes.append(float(lng))
        except Exception:
            latitudes.append(np.nan)
            longitudes.append(np.nan)

    df["lat"] = latitudes
    df["lng"] = longitudes

    # missing indicators
    df["temp_missing"] = df["Temperature"].isna().astype(int)
    df["road_missing"] = df["RoadType"].isna().astype(int)
    df["weather_missing"] = df["Weather"].isna().astype(int)

    # simple imputation
    df["Temperature"] = df["Temperature"].fillna(df["Temperature"].median())
    df["RoadType"] = df["RoadType"].fillna("Unknown")
    df["Weather"] = df["Weather"].fillna("Unknown")

    # optional interaction feature
    df["gh_hour"] = df["geohash"].astype(str) + "_" + df["hour"].astype(str)

    # categoricals
    cat_cols = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather", "gh_hour"]
    for col in cat_cols:
        df[col] = df[col].astype("category")

    return df

train = engineer(train)
test = engineer(test)

FEATURES = [
    "geohash",
    "day",
    "hour",
    "minute",
    "minute_block",
    "time_slot",
    "lat",
    "lng",
    "RoadType",
    "NumberofLanes",
    "LargeVehicles",
    "Landmarks",
    "Temperature",
    "Weather",
    "temp_missing",
    "road_missing",
    "weather_missing",
    "gh_hour",
]

CAT_COLS = ["geohash", "RoadType", "LargeVehicles", "Landmarks", "Weather", "gh_hour"]
TARGET = "demand"

X = train[FEATURES].copy()
y = train[TARGET].copy()
X_test = test[FEATURES].copy()


In [ ]:
# ============================================
# 5) Leak-safe target encoding
# ============================================
def add_target_encoding(X_tr, y_tr, X_val, X_te, cols, smooth=20):
    """
    Fit target encodings on X_tr/y_tr only.
    Apply to X_tr, X_val, X_te using the same mapping.
    Unseen categories fall back to global mean.
    """
    X_tr = X_tr.copy()
    X_val = X_val.copy()
    X_te = X_te.copy()

    global_mean = y_tr.mean()

    for col in cols:
        stats = (
            pd.DataFrame({"key": X_tr[col].astype(str), "target": y_tr.values})
            .groupby("key")["target"]
            .agg(["mean", "count"])
        )

        enc = (stats["mean"] * stats["count"] + global_mean * smooth) / (stats["count"] + smooth)

        te_col = f"te_{col}"
        X_tr[te_col] = X_tr[col].astype(str).map(enc).fillna(global_mean)
        X_val[te_col] = X_val[col].astype(str).map(enc).fillna(global_mean)
        X_te[te_col] = X_te[col].astype(str).map(enc).fillna(global_mean)

    return X_tr, X_val, X_te

TE_COLS = ["geohash", "hour", "gh_hour"]

In [ ]:
# ============================================
# 6) Cross-validation setup
# ============================================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))
pred_lgb = np.zeros(len(test))
pred_cat = np.zeros(len(test))

In [ ]:
# ============================================
# 7) Train folds
# ============================================
for fold, (tr_idx, val_idx) in enumerate(kf.split(X), 1):
    X_tr = X.iloc[tr_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_tr = y.iloc[tr_idx].copy()
    y_val = y.iloc[val_idx].copy()
    X_te = X_test.copy()

    # target encoding
    X_tr, X_val, X_te = add_target_encoding(X_tr, y_tr, X_val, X_te, TE_COLS, smooth=20)

    # update feature list after TE columns are added
    fold_features = list(X_tr.columns)

    # categorical feature indices for both models
    cat_idx = [i for i, c in enumerate(fold_features) if c in CAT_COLS]

    # =========================
    # LightGBM
    # =========================
    lgb_train = lgb.Dataset(
        X_tr,
        label=y_tr,
        categorical_feature=cat_idx,
        free_raw_data=False,
    )
    lgb_valid = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_idx,
        reference=lgb_train,
        free_raw_data=False,
    )

    lgb_params = {
        "objective": "regression",
        "metric": "rmse",
        "learning_rate": 0.05,
        "num_leaves": 255,
        "min_child_samples": 20,
        "feature_fraction": 0.9,
        "bagging_fraction": 0.8,
        "bagging_freq": 1,
        "lambda_l1": 0.0,
        "lambda_l2": 0.0,
        "verbosity": -1,
        "seed": RANDOM_STATE,
    }

    lgb_model = lgb.train(
        lgb_params,
        lgb_train,
        num_boost_round=5000,
        valid_sets=[lgb_valid],
        callbacks=[lgb.early_stopping(100, verbose=False)],
    )

    oof_lgb[val_idx] = lgb_model.predict(X_val, num_iteration=lgb_model.best_iteration)
    pred_lgb += lgb_model.predict(X_te, num_iteration=lgb_model.best_iteration) / N_FOLDS

    # =========================
    # CatBoost
    # =========================
    cat_model = cb.CatBoostRegressor(
        iterations=5000,
        learning_rate=0.05,
        depth=8,
        loss_function="RMSE",
        eval_metric="R2",
        random_seed=RANDOM_STATE,
        verbose=0,
        early_stopping_rounds=100,
    )

    cat_model.fit(
        X_tr,
        y_tr,
        cat_features=cat_idx,
        eval_set=(X_val, y_val),
        use_best_model=True,
    )

    oof_cat[val_idx] = cat_model.predict(X_val)
    pred_cat += cat_model.predict(X_te) / N_FOLDS

    fold_r2_lgb = r2_score(y_val, oof_lgb[val_idx])
    fold_r2_cat = r2_score(y_val, oof_cat[val_idx])

    print(f"Fold {fold:02d} | LGB R2: {fold_r2_lgb:.5f} | CAT R2: {fold_r2_cat:.5f}")

Fold 01 | LGB R2: 0.94147 | CAT R2: 0.94362
Fold 02 | LGB R2: 0.94077 | CAT R2: 0.93636
Fold 03 | LGB R2: 0.94091 | CAT R2: 0.94425
Fold 04 | LGB R2: 0.93670 | CAT R2: 0.93745
Fold 05 | LGB R2: 0.94596 | CAT R2: 0.94798


In [ ]:
# ============================================
# 8) Ensemble and evaluation
# ============================================
final_oof = 0.5 * oof_lgb + 0.5 * oof_cat
final_pred = 0.5 * pred_lgb + 0.5 * pred_cat

# demand appears bounded in [0, 1], so clipping is reasonable
final_pred = np.clip(final_pred, 0, 1)

overall_r2 = r2_score(y, final_oof)
print(f"\nOOF R2: {overall_r2:.6f}")
print(f"Competition-style score: {max(0, 100 * overall_r2):.4f}")

# ============================================
sub = pd.DataFrame({
    "Index": test["Index"],
    "demand": final_pred
})

sub.to_csv("submission.csv", index=False)
print("Saved submission.csv with shape:", sub.shape)



OOF R2: 0.944538
Competition-style score: 94.4538
Saved submission.csv with shape: (41778, 2)
